# GO Enrichment — Genes específicos de tratamiento

Análisis de sobrerrepresentación (ORA) con Fisher exacto + FDR BH sobre:
1. Genes bloqueados por Ascórbico (sig en Control, NS en Asc)
2. Genes bloqueados por Etileno (sig en Control, NS en Eth)
3. Genes específicos de Ascórbico en ER vs Endo (sig solo en Asc)
4. Genes específicos de Etileno en ER vs Endo (sig solo en Eth)

## 1. Configuración

In [ ]:
# AJUSTAR ESTAS RUTAS
GO_FILE       = 'pdulcis_Nonpareil_v1_0_genes2Go.xlsx'
ECO_ENDO_FILE = 'DEG_EcoD_vs_EndoD_by_treatment.csv'
ER_ENDO_FILE  = 'DEG_EndoR_vs_Endo_by_treatment.csv'
ECO_ER_FILE   = 'DEG_EcoD_vs_EndoR_by_treatment.csv'
OUTPUT_DIR    = './GO_treatment_results'

PADJ_THRESHOLD   = 0.05
MIN_INTERSECTION = 3

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('OK')

## 2. Librerías

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
print('OK')

## 3. Cargar anotaciones GO

In [ ]:
go_raw = pd.read_excel(GO_FILE, header=2)
go_raw.columns = ['gene_id', 'go_term', 'description']
go_raw = go_raw.dropna(subset=['gene_id','go_term'])
go_raw['PRUDU_ID']  = go_raw['gene_id'].str.extract(r'(PRUDU\d+)')
go_raw['namespace'] = go_raw['description'].str.split(':').str[0].str.strip()
go_raw['go_name']   = go_raw['description'].str.split(':', n=1).str[1].str.strip()
go_raw = go_raw.dropna(subset=['PRUDU_ID'])

universe = set(go_raw['PRUDU_ID'].unique())
go2gene  = go_raw.groupby('go_term')['PRUDU_ID'].apply(set).to_dict()
go_info  = go_raw.drop_duplicates('go_term').set_index('go_term')[['go_name','namespace']].to_dict('index')

print(f'Genes en universo GO: {len(universe):,}')
print(f'Términos GO únicos:   {len(go2gene):,}')

## 4. Cargar DEGs de tratamiento

In [ ]:
eco_endo = pd.read_csv(ECO_ENDO_FILE, index_col=0)
er_endo  = pd.read_csv(ER_ENDO_FILE,  index_col=0)
eco_er   = pd.read_csv(ECO_ER_FILE,   index_col=0)

# ── Definir conjuntos de genes para cada análisis ────────────────────────────

# 1. Bloqueados por Ascórbico en Eco vs Endo
#    sig en Control PERO NS en Ascórbico
blocked_asc_eco = set(eco_endo[
    eco_endo['reg_Control'].isin(['UP','DOWN']) &
    (eco_endo['reg_Ascorbic'] == 'NS')
].index)

# 2. Bloqueados por Etileno en Eco vs Endo
blocked_eth_eco = set(eco_endo[
    eco_endo['reg_Control'].isin(['UP','DOWN']) &
    (eco_endo['reg_Ethylene'] == 'NS')
].index)

# 3. Específicos de Ascórbico en ER vs Endo
#    sig en Asc PERO NS en Control
spec_asc_er = set(er_endo[
    er_endo['reg_Ascorbic'].isin(['UP','DOWN']) &
    (er_endo['reg_Control'] == 'NS')
].index)

# 4. Específicos de Etileno en ER vs Endo
spec_eth_er = set(er_endo[
    er_endo['reg_Ethylene'].isin(['UP','DOWN']) &
    (er_endo['reg_Control'] == 'NS')
].index)

# 5. Comunes a ambos tratamientos en ER vs Endo (específicos de ambos)
spec_both_er = spec_asc_er & spec_eth_er

QUERY_SETS = {
    'Bloqueados_Asc_EcoEndo':  blocked_asc_eco,
    'Bloqueados_Eth_EcoEndo':  blocked_eth_eco,
    'Especificos_Asc_EREndo':  spec_asc_er,
    'Especificos_Eth_EREndo':  spec_eth_er,
    'Especificos_Ambos_EREndo': spec_both_er,
}

print('Conjuntos de genes:')
for name, genes in QUERY_SETS.items():
    in_go = genes & universe
    print(f'  {name:35s}: {len(genes):4d} genes ({len(in_go):4d} con GO)')

## 5. Función ORA

In [ ]:
def run_ora(query_genes, universe_genes, go2gene, go_info, min_intersection=3):
    query   = query_genes & universe_genes
    N       = len(universe_genes)
    n_query = len(query)
    if n_query == 0:
        return pd.DataFrame()

    results = []
    for go_term, go_genes in go2gene.items():
        in_query = query & go_genes
        if len(in_query) < min_intersection:
            continue
        a = len(in_query)
        b = len(go_genes - query)
        c = n_query - a
        d = N - n_query - b
        _, pval = fisher_exact([[a, b], [c, d]], alternative='greater')
        info = go_info.get(go_term, {})
        results.append({
            'go_term':              go_term,
            'go_name':              info.get('go_name', ''),
            'namespace':            info.get('namespace', ''),
            'n_universe':           len(go_genes & universe_genes),
            'n_query':              n_query,
            'intersection':         a,
            'fold_enrichment':      (a/n_query) / (len(go_genes & universe_genes)/N),
            'intersection_genes':   ';'.join(sorted(in_query)),
            'pvalue':               pval,
        })

    if not results:
        return pd.DataFrame()

    df = pd.DataFrame(results)
    _, padj, _, _ = multipletests(df['pvalue'], method='fdr_bh')
    df['padj'] = padj
    return df.sort_values('padj').reset_index(drop=True)

print('Función ORA definida OK')

## 6. Ejecutar enrichment para todos los conjuntos

In [ ]:
all_ora = {}

for name, genes in QUERY_SETS.items():
    print(f'\n{name}...')
    res = run_ora(genes, universe, go2gene, go_info,
                  min_intersection=MIN_INTERSECTION)
    if len(res) == 0:
        print('  Sin resultados')
        all_ora[name] = pd.DataFrame()
        continue

    sig = res[res['padj'] < PADJ_THRESHOLD]
    print(f'  Términos testados: {len(res)} | Significativos (FDR<{PADJ_THRESHOLD}): {len(sig)}')

    if len(sig) > 0:
        print(f'  Top 5:')
        for _, r in sig.head(5).iterrows():
            print(f'    {r["go_term"]} {r["go_name"][:50]} '
                  f'(n={r["intersection"]}, FE={r["fold_enrichment"]:.1f}, padj={r["padj"]:.3f})')
    else:
        print(f'  Top 5 (sin FDR):')
        for _, r in res.head(5).iterrows():
            print(f'    {r["go_term"]} {r["go_name"][:50]} '
                  f'(n={r["intersection"]}, padj={r["padj"]:.3f})')

    outfile = os.path.join(OUTPUT_DIR, f'GO_ORA_{name}.csv')
    res.to_csv(outfile, index=False)
    all_ora[name] = res

print('\nFINALIZADO')

## 7. Figura combinada — dot plot por conjunto

In [ ]:
NS_COLORS = {
    'Biological Process': '#1a9641',
    'Molecular Function': '#2166ac',
    'Cellular Component': '#d95f02',
}

# Filtrar solo conjuntos con resultados significativos
sets_with_sig = {k: v for k, v in all_ora.items()
                 if len(v) > 0 and (v['padj'] < PADJ_THRESHOLD).any()}

if not sets_with_sig:
    print('Ningún conjunto tiene términos significativos con FDR<0.05.')
    print('Generando figura con top términos sin filtro de FDR...')
    sets_with_sig = {k: v for k, v in all_ora.items() if len(v) > 0}
    use_fdr = False
else:
    use_fdr = True

n_sets = len(sets_with_sig)
if n_sets == 0:
    print('Sin datos para graficar.')
else:
    fig, axes = plt.subplots(1, n_sets,
                             figsize=(6*n_sets, 8),
                             facecolor='white')
    if n_sets == 1:
        axes = [axes]

    fig.suptitle('Enriquecimiento GO — Genes modulados por tratamiento\nP. dulcis cv. Nonpareil',
                 fontsize=12, fontweight='bold')

    for ax, (name, res) in zip(axes, sets_with_sig.items()):
        if use_fdr:
            plot_df = res[res['padj'] < PADJ_THRESHOLD].nsmallest(15, 'padj').copy()
        else:
            plot_df = res.nsmallest(15, 'pvalue').copy()

        if len(plot_df) == 0:
            ax.text(0.5, 0.5, 'Sin resultados', ha='center', va='center',
                    transform=ax.transAxes)
            continue

        plot_df = plot_df.sort_values('padj', ascending=False)
        plot_df['log10p'] = -np.log10(plot_df['padj'].clip(1e-300))
        plot_df['color']  = plot_df['namespace'].map(NS_COLORS).fillna('#888888')

        ax.barh(range(len(plot_df)), plot_df['log10p'],
                color=plot_df['color'], alpha=0.8,
                edgecolor='white', linewidth=0.5)

        # Puntos con tamaño = intersección
        sizes = (plot_df['intersection'] / plot_df['intersection'].max() * 200) + 40
        ax.scatter(plot_df['log10p'] + 0.05, range(len(plot_df)),
                   s=sizes, c=plot_df['color'], zorder=3,
                   edgecolors='white', linewidths=0.5, alpha=0.9)

        ax.set_yticks(range(len(plot_df)))
        labels = [f"{r['go_term']} — {r['go_name'][:40]}" for _, r in plot_df.iterrows()]
        ax.set_yticklabels(labels, fontsize=7.5)
        ax.set_xlabel('−log₁₀(padj)', fontsize=9)
        ax.axvline(-np.log10(PADJ_THRESHOLD), color='#888888',
                   linewidth=1, linestyle='--', alpha=0.6)
        ax.set_title(name.replace('_', ' '), fontsize=9, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        for i, (_, r) in enumerate(plot_df.iterrows()):
            ax.text(0.05, i, f"n={r['intersection']}",
                    va='center', ha='left', fontsize=6.5,
                    color='white', fontweight='bold')

    legend_elements = [mpatches.Patch(facecolor=c, label=ns, alpha=0.85)
                       for ns, c in NS_COLORS.items()]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.03))

    plt.tight_layout()
    outfig = os.path.join(OUTPUT_DIR, 'GO_treatment_dotplot.png')
    plt.savefig(outfig, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Figura guardada: {outfig}')

## 8. Comparación entre tratamientos — términos compartidos

In [ ]:
# Términos significativos por conjunto
sig_by_set = {}
for name, res in all_ora.items():
    if len(res) > 0:
        sig_by_set[name] = set(res[res['padj'] < PADJ_THRESHOLD]['go_term'])
    else:
        sig_by_set[name] = set()

print('Términos GO significativos por conjunto:')
for name, terms in sig_by_set.items():
    print(f'  {name:40s}: {len(terms)}')

# Términos compartidos entre Asc y Eth bloqueados
shared_blocked = sig_by_set.get('Bloqueados_Asc_EcoEndo', set()) & \
                 sig_by_set.get('Bloqueados_Eth_EcoEndo', set())
print(f'\nTérminos compartidos (bloqueados Asc y Eth): {len(shared_blocked)}')
if shared_blocked and len(all_ora.get('Bloqueados_Asc_EcoEndo', pd.DataFrame())) > 0:
    res_asc = all_ora['Bloqueados_Asc_EcoEndo']
    shared_df = res_asc[res_asc['go_term'].isin(shared_blocked)]
    for _, r in shared_df.head(10).iterrows():
        print(f'  {r["go_term"]} {r["go_name"][:60]}')

# Términos compartidos entre Asc y Eth específicos ER
shared_spec = sig_by_set.get('Especificos_Asc_EREndo', set()) & \
              sig_by_set.get('Especificos_Eth_EREndo', set())
print(f'\nTérminos compartidos (específicos Asc y Eth ER vs Endo): {len(shared_spec)}')
if shared_spec and len(all_ora.get('Especificos_Asc_EREndo', pd.DataFrame())) > 0:
    res_asc_er = all_ora['Especificos_Asc_EREndo']
    shared_er_df = res_asc_er[res_asc_er['go_term'].isin(shared_spec)]
    for _, r in shared_er_df.head(10).iterrows():
        print(f'  {r["go_term"]} {r["go_name"][:60]}')

## 9. Dot plots por conjunto

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import os

# Configuración inicial
NS_COLORS = {
    'Biological Process': '#1a9641',
    'Molecular Function': '#2166ac',
    'Cellular Component': '#d95f02',
}
PADJ_THRESHOLD = 0.05
TOP_N          = 15   # términos a mostrar por conjunto

# Cargar resultados 
# Usa FDR<0.05 si hay; si no, los top N por p-value
sets_to_plot = {
    'Bloqueados por\nAscórbico\n(Eco vs Endo)':    'GO_ORA_Bloqueados_Asc_EcoEndo.csv',
    'Bloqueados por\nEtileno\n(Eco vs Endo)':       'GO_ORA_Bloqueados_Eth_EcoEndo.csv',
    'Específicos\nAscórbico\n(ER vs Endo)':         'GO_ORA_Especificos_Asc_EREndo.csv',
    'Específicos\nEtileno\n(ER vs Endo)':           'GO_ORA_Especificos_Eth_EREndo.csv',
}

loaded = {}
for label, fname in sets_to_plot.items():
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        if len(df) > 0:
            loaded[label] = df
            sig = (df['padj'] < PADJ_THRESHOLD).sum()
            print(f'{label.replace(chr(10)," "):45s}: {len(df)} términos, {sig} sig.')

if not loaded:
    print('No hay archivos CSV. Ejecuta primero la celda 6.')

#FIGURA A: Dot plot combinado (todos los conjuntos en un panel) 
if loaded:
    # Seleccionar top N por padj de cada conjunto
    frames = []
    for label, df in loaded.items():
        sig = df[df['padj'] < PADJ_THRESHOLD]
        top = sig.nsmallest(TOP_N, 'padj') if len(sig) >= 3 else df.nsmallest(TOP_N, 'pvalue')
        top = top.copy()
        top['query_set'] = label
        frames.append(top)

    if frames:
        all_top = pd.concat(frames, ignore_index=True)

        # Una figura por conjunto
        n_sets = len(loaded)
        fig, axes = plt.subplots(1, n_sets, figsize=(6*n_sets, 9), facecolor='white')
        if n_sets == 1:
            axes = [axes]

        fig.suptitle('Enriquecimiento funcional GO — Genes modulados por tratamiento\n'
                     'Prunus dulcis cv. Nonpareil | Fisher exacto + FDR BH',
                     fontsize=12, fontweight='bold')

        for ax, (label, df) in zip(axes, loaded.items()):
            sig = df[df['padj'] < PADJ_THRESHOLD]
            use_fdr = len(sig) >= 3
            plot_df = sig.nsmallest(TOP_N, 'padj') if use_fdr else df.nsmallest(TOP_N, 'pvalue')
            plot_df = plot_df.sort_values('padj', ascending=False).reset_index(drop=True)

            # Calcular gene ratio
            plot_df['gene_ratio'] = plot_df['intersection'] / plot_df['n_query']
            plot_df['log10p']     = -np.log10(plot_df['padj'].clip(1e-300))
            plot_df['color']      = plot_df['namespace'].map(NS_COLORS).fillna('#888888')

            # Normalizar tamaño de puntos
            size_min, size_max = 40, 300
            n_vals = plot_df['intersection'].values
            if n_vals.max() > n_vals.min():
                sizes = size_min + (n_vals - n_vals.min()) / (n_vals.max() - n_vals.min()) * (size_max - size_min)
            else:
                sizes = np.full(len(plot_df), (size_min + size_max) / 2)

            # Scatter plot
            sc = ax.scatter(plot_df['log10p'], range(len(plot_df)),
                           c=plot_df['color'], s=sizes,
                           alpha=0.85, zorder=3,
                           edgecolors='white', linewidths=0.5)

            # Línea punteada desde 0 hasta el punto
            for yi, (_, row) in enumerate(plot_df.iterrows()):
                ax.plot([0, row['log10p']], [yi, yi],
                        color=row['color'], linewidth=0.8,
                        alpha=0.4, zorder=1)

            # Línea de umbral FDR
            ax.axvline(-np.log10(PADJ_THRESHOLD), color='#888888',
                       linewidth=1, linestyle='--', alpha=0.6)

            # Etiquetas Y
            ax.set_yticks(range(len(plot_df)))
            y_labels = [f"{r['go_term']} {r['go_name'][:35]}" for _, r in plot_df.iterrows()]
            ax.set_yticklabels(y_labels, fontsize=7.5)

            # Anotar n genes en cada punto
            for yi, (_, row) in enumerate(plot_df.iterrows()):
                ax.text(row['log10p'] + 0.05, yi,
                        f" n={int(row['intersection'])}",
                        va='center', ha='left', fontsize=6.5, color='#333333')

            ax.set_xlabel('−log₁₀(padj)', fontsize=9)
            ax.set_title(label.replace('\n', ' '), fontsize=9, fontweight='bold')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_ylim(-0.8, len(plot_df) - 0.2)
            ax.yaxis.grid(True, alpha=0.15, linestyle='--')
            ax.set_axisbelow(True)

            if not use_fdr:
                ax.text(0.98, 0.02, 'Sin términos FDR<0.05\n(mostrando top por p-value)',
                        transform=ax.transAxes, ha='right', va='bottom',
                        fontsize=7, color='#888888', style='italic')

        # Leyenda namespace
        legend_ns = [mpatches.Patch(facecolor=c, label=ns, alpha=0.85)
                     for ns, c in NS_COLORS.items()]
        # Leyenda tamaño puntos
        for size_val in [5, 20, 50]:
            legend_ns.append(plt.scatter([], [], c='#888888', s=size_val*4,
                             alpha=0.7, label=f'n={size_val}'))
        fig.legend(handles=legend_ns, loc='lower center', ncol=6,
                   fontsize=8, framealpha=0.9, bbox_to_anchor=(0.5, -0.05))

        plt.tight_layout()
        outfig = os.path.join(OUTPUT_DIR, 'GO_treatment_dotplot_final.png')
        plt.savefig(outfig, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f'Figura guardada: {outfig}')

# FIGURA B: Solo Biological Process — los más informativos 
if loaded:
    fig, axes = plt.subplots(1, n_sets, figsize=(6*n_sets, 8), facecolor='white')
    if n_sets == 1:
        axes = [axes]

    fig.suptitle('Enriquecimiento GO — Biological Process\n'
                 'Genes modulados por tratamiento | Prunus dulcis cv. Nonpareil',
                 fontsize=12, fontweight='bold')

    for ax, (label, df) in zip(axes, loaded.items()):
        bp = df[df['namespace'] == 'Biological Process']
        sig = bp[bp['padj'] < PADJ_THRESHOLD]
        use_fdr = len(sig) >= 3
        plot_df = sig.nsmallest(15, 'padj') if use_fdr else bp.nsmallest(15, 'pvalue')
        plot_df = plot_df.sort_values('padj', ascending=False).reset_index(drop=True)

        if len(plot_df) == 0:
            ax.text(0.5, 0.5, 'Sin términos BP', ha='center', va='center',
                    transform=ax.transAxes)
            continue

        plot_df['log10p'] = -np.log10(plot_df['padj'].clip(1e-300))
        plot_df['fold_enrichment'] = plot_df['fold_enrichment'].clip(0, 20)

        # Color por fold enrichment
        cmap   = plt.cm.YlOrRd
        norm   = plt.Normalize(plot_df['fold_enrichment'].min(),
                               plot_df['fold_enrichment'].max())
        colors = [cmap(norm(fe)) for fe in plot_df['fold_enrichment']]

        n_vals = plot_df['intersection'].values
        if n_vals.max() > n_vals.min():
            sizes = 40 + (n_vals - n_vals.min()) / (n_vals.max() - n_vals.min()) * 260
        else:
            sizes = np.full(len(plot_df), 150)

        sc = ax.scatter(plot_df['log10p'], range(len(plot_df)),
                        c=plot_df['fold_enrichment'], cmap='YlOrRd',
                        s=sizes, vmin=1, vmax=plot_df['fold_enrichment'].max(),
                        alpha=0.9, zorder=3, edgecolors='#333333', linewidths=0.4)

        for yi, (_, row) in enumerate(plot_df.iterrows()):
            ax.plot([0, row['log10p']], [yi, yi],
                    color='#cccccc', linewidth=0.8, zorder=1)
            ax.text(row['log10p'] + 0.05, yi,
                    f" n={int(row['intersection'])}",
                    va='center', ha='left', fontsize=6.5, color='#333333')

        ax.axvline(-np.log10(PADJ_THRESHOLD), color='#888888',
                   linewidth=1, linestyle='--', alpha=0.6)

        ax.set_yticks(range(len(plot_df)))
        y_labels = [f"{r['go_name'][:45]}" for _, r in plot_df.iterrows()]
        ax.set_yticklabels(y_labels, fontsize=7.5)
        ax.set_xlabel('−log₁₀(padj)', fontsize=9)
        ax.set_title(label.replace('\n', ' '), fontsize=9, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_ylim(-0.8, len(plot_df) - 0.2)
        ax.yaxis.grid(True, alpha=0.15, linestyle='--')
        ax.set_axisbelow(True)

        plt.colorbar(sc, ax=ax, label='Fold enrichment', shrink=0.4, pad=0.01)

        if not use_fdr:
            ax.text(0.98, 0.02, 'Sin términos FDR<0.05',
                    transform=ax.transAxes, ha='right', va='bottom',
                    fontsize=7, color='#888888', style='italic')

    plt.tight_layout()
    outfig_bp = os.path.join(OUTPUT_DIR, 'GO_treatment_BP_dotplot.png')
    plt.savefig(outfig_bp, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Figura BP guardada: {outfig_bp}')
